# Quản lý thay đổi thiết kế để hạn chế chậm tiến độ

Notebook v2 phục vụ bài tập cuối kỳ môn Data Storytelling. Các kết quả được diễn giải trong phạm vi bộ dữ liệu dự án đi kèm.

**Thông điệp trung tâm:** Các yêu cầu thay đổi thiết kế phát sinh sau mốc chốt có liên hệ với số ngày chậm cao hơn. Nhóm đề xuất Trưởng Ban Quản lý dự án phê duyệt chương trình thí điểm gồm mốc chốt thiết kế, tổ kiểm soát thay đổi và thời hạn xử lý 48 giờ trước khi áp dụng rộng.

**Người ra quyết định chính:** Trưởng Ban Quản lý dự án.


## Storyboard

1. Bối cảnh — quy mô trễ tiến độ.
2. Điểm mở đầu — khác biệt giữa nhóm có và không thay đổi thiết kế.
3. Bằng chứng — xu hướng theo số lần thay đổi và các kiểm tra tính nhất quán.
4. Kết luận chính — mức tăng ngày chậm đi cùng mỗi lần thay đổi.
5. Hành động — chương trình thí điểm và phân tích tác động.

Mỗi biểu đồ trả lời một câu hỏi; các hình kiểm tra chi tiết không tạo thêm hướng kể chuyện mới.


## 1. Thiết lập môi trường và ngôn ngữ trực quan

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
SOURCE = ROOT / "du_lieu_tien_do_du_an.csv"
CHARTS = ROOT / "charts"
CHARTS.mkdir(parents=True, exist_ok=True)

RED = "#B83A32"
NAVY = "#23363D"
GREEN = "#2F6B5C"
GRAY = "#AEB7BB"
MUTED = "#617178"

sns.set_theme(style="white", font_scale=1.04)
plt.rcParams.update({
    "figure.dpi": 160, "font.family": "DejaVu Sans",
    "axes.titleweight": "bold", "axes.titlesize": 15,
    "axes.labelsize": 11, "axes.edgecolor": "#D6DEE1",
    "xtick.color": NAVY, "ytick.color": NAVY, "text.color": NAVY,
})


## 2. Hàm xử lý dữ liệu và tạo bộ biểu đồ v2

In [ ]:
def currency(v):
    if pd.isna(v): return np.nan
    s = str(v).lower().replace(" ", "").replace(",", ".")
    m = re.search(r"\d+(?:\.\d+)?", s)
    if not m: return np.nan
    n = float(m.group())
    return n / 1000 if "triệu" in s or n > 1000 else n


def vi_num(v, digits=1):
    """Định dạng số theo quy ước tiếng Việt: dấu phẩy thập phân, dấu chấm hàng nghìn."""
    return f"{v:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")


def vi_int(v):
    return f"{v:,.0f}".replace(",", ".")


def load_data():
    d = pd.read_csv(SOURCE)
    for c in ["Ngày bắt đầu", "Ngày kết thúc theo hợp đồng", "Ngày kết thúc thực tế"]:
        d[c] = pd.to_datetime(d[c], dayfirst=True, errors="coerce")
    d["Giá trị hợp đồng (Tỷ VNĐ)"] = d["Giá trị hợp đồng"].map(currency)
    d["Trạng thái"] = np.where(d["Số ngày chậm"] > 0, "Trễ hạn", "Đúng/Sớm hạn")
    d["Thiết kế"] = np.where(d["Số lần thay đổi thiết kế"] > 0, "Có thay đổi", "Không thay đổi")
    d["Năm bắt đầu"] = d["Ngày bắt đầu"].dt.year
    d["Nhóm thay đổi"] = pd.cut(
        d["Số lần thay đổi thiết kế"], [-.1, 0, 1, 2, 4, 6, 8],
        labels=["0", "1", "2", "3–4", "5–6", "7–8"], include_lowest=True
    )
    d["Nhóm phân phối"] = pd.cut(
        d["Số lần thay đổi thiết kế"], [-.1, 0, 2, 5, np.inf],
        labels=["0", "1–2", "3–5", ">5"], include_lowest=True
    )
    return d


def title(ax, main, sub=""):
    ax.set_title(main, loc="left", pad=22, color=NAVY)
    if sub:
        ax.text(0, 1.015, sub, transform=ax.transAxes, color=MUTED, fontsize=10, va="bottom")


def clean(ax, grid="x"):
    ax.spines[["top", "right"]].set_visible(False)
    if grid:
        ax.grid(axis=grid, color="#DFE6E8", linewidth=.8, alpha=.9)
        ax.set_axisbelow(True)


def save(fig, filename):
    fig.tight_layout()
    fig.savefig(CHARTS / filename, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def coefficients(d):
    features = ["Số lần thay đổi thiết kế", "Số ngày mưa", "Số lần thiếu vật tư", "Giá trị hợp đồng (Tỷ VNĐ)"]
    m = d[features + ["Số ngày chậm"]].dropna()
    X = (m[features] - m[features].mean()) / m[features].std(ddof=0)
    y = (m["Số ngày chậm"] - m["Số ngày chậm"].mean()) / m["Số ngày chậm"].std(ddof=0)
    A = np.column_stack([np.ones(len(X)), X])
    beta = np.linalg.lstsq(A, y, rcond=None)[0]
    pred = A @ beta
    r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return pd.DataFrame({"Yếu tố": features, "Hệ số": beta[1:]}).sort_values("Hệ số"), r2


def make_charts(d):
    # 01 — thanh tỷ lệ tổng thể
    late_pct = int(round((d["Số ngày chậm"] > 0).mean() * 100))
    fig, ax = plt.subplots(figsize=(11, 4.6))
    ax.barh([0], [late_pct], color=RED, height=.5, label="Trễ hạn")
    ax.barh([0], [100-late_pct], left=[late_pct], color=GREEN, height=.5, label="Đúng hoặc sớm hạn")
    ax.text(late_pct/2, 0, f"Trễ hạn\n{late_pct}%", ha="center", va="center", color="white", fontsize=16, fontweight="bold")
    ax.text(late_pct+(100-late_pct)/2, 0, f"Đúng hoặc sớm hạn\n{100-late_pct}%", ha="center", va="center", color="white", fontsize=13, fontweight="bold")
    ax.set_xlim(0, 100); ax.set_yticks([]); ax.set_xlabel("Tỷ lệ dự án (%)")
    title(ax, f"{late_pct}% dự án hoàn thành sau thời hạn hợp đồng", "Tỷ lệ được tính trên toàn bộ 6.000 dự án")
    clean(ax, "x")
    save(fig, "01_thanh_ty_le_tien_do.png")

    # 02 — hai biểu đồ cột so sánh hai nhóm
    g = d.groupby("Thiết kế").agg(
        ty_le_tre=("Số ngày chậm", lambda s: (s > 0).mean() * 100),
        cham_tb=("Số ngày chậm", "mean")
    ).reindex(["Không thay đổi", "Có thay đổi"])
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    specs = [("ty_le_tre", "Tỷ lệ trễ hạn", "%", (0, 100)), ("cham_tb", "Số ngày chậm trung bình", " ngày", (-3, 12))]
    for ax, (col, ttl, unit, lim) in zip(axes, specs):
        vals = g[col].values
        bars = ax.bar(["Không thay đổi", "Có thay đổi"], vals, color=[GRAY, RED], width=.6)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, val + (.8 if val >= 0 else -.8), f"{vi_num(val)}{unit}", ha="center", va="bottom" if val >= 0 else "top", fontweight="bold")
        ax.set_ylim(*lim); ax.set_ylabel(ttl); ax.axhline(0, color=NAVY, lw=.8)
        clean(ax, "y")
    fig.suptitle("Dự án có thay đổi thiết kế trễ nhiều hơn", x=.06, ha="left", fontsize=16, fontweight="bold")
    save(fig, "02_cot_so_sanh_hai_nhom.png")

    # 03 — đường xu hướng theo mức thay đổi
    level_summary = d.groupby("Nhóm thay đổi", observed=False).agg(cham_tb=("Số ngày chậm", "mean"), n=("STT", "count")).reset_index()
    x = np.arange(len(level_summary))
    fig, ax = plt.subplots(figsize=(11.5, 5.8))
    ax.plot(x, level_summary.cham_tb, color=RED, lw=3, marker="o", markersize=9)
    for xi, y, n in zip(x, level_summary.cham_tb, level_summary.n):
        ax.text(xi, y + 5, vi_num(y), ha="center", fontweight="bold")
        ax.text(xi, y - 9, f"n={vi_int(n)}", ha="center", fontsize=9, color=MUTED)
    ax.set_xticks(x, level_summary["Nhóm thay đổi"].astype(str)); ax.axhline(0, color=NAVY, lw=.8); ax.set_ylim(-10, 45)
    ax.set_ylabel("Số ngày chậm trung bình"); ax.set_xlabel("Số lần thay đổi thiết kế")
    title(ax, "Độ trễ tăng theo từng mức thay đổi thiết kế", "Số dự án ở từng nhóm được ghi ngay dưới điểm dữ liệu")
    clean(ax, "y")
    save(fig, "03_duong_muc_thay_doi.png")

    # 04 — biểu đồ phân tán với đường xu hướng
    slope, intercept = np.polyfit(d["Số lần thay đổi thiết kế"], d["Số ngày chậm"], 1)
    corr = d["Số lần thay đổi thiết kế"].corr(d["Số ngày chậm"])
    fig, ax = plt.subplots(figsize=(11.5, 6))
    max_design = int(d["Số lần thay đổi thiết kế"].max())
    jitter = np.random.default_rng(17).normal(0, .07, len(d))
    ax.scatter(d["Số lần thay đổi thiết kế"] + jitter, d["Số ngày chậm"], s=10, alpha=.12, color=RED, edgecolors="none")
    xx = np.linspace(0, max_design, 200); ax.plot(xx, intercept + slope * xx, color=NAVY, lw=2.2, linestyle="--")
    ax.text(.97, .08, f"Độ dốc: {vi_num(slope)} ngày/lần\nr = {vi_num(corr, 3)}", transform=ax.transAxes,
            ha="right", fontweight="bold", color=RED, bbox=dict(facecolor="white", edgecolor=RED, boxstyle="round,pad=.45"))
    ax.set_xlabel("Số lần thay đổi thiết kế"); ax.set_ylabel("Số ngày chậm")
    title(ax, "Số ngày chậm có xu hướng tăng khi số lần thay đổi tăng", "Mỗi điểm là một dự án; đường nét đứt thể hiện xu hướng trung bình")
    clean(ax, "y")
    save(fig, "04_phan_tan_xu_huong.png")

    # 05 — thanh ngang hệ số chuẩn hóa
    coef, r2 = coefficients(d)
    fig, ax = plt.subplots(figsize=(10.5, 5.5))
    colors = [RED if name == "Số lần thay đổi thiết kế" else GRAY for name in coef["Yếu tố"]]
    bars = ax.barh(coef["Yếu tố"], coef["Hệ số"], color=colors, height=.58)
    for bar, val, name in zip(bars, coef["Hệ số"], coef["Yếu tố"]):
        ax.text(val + .012, bar.get_y()+bar.get_height()/2, vi_num(val, 3), va="center", fontweight="bold" if name == "Số lần thay đổi thiết kế" else "normal")
    ax.axvline(0, color=NAVY, lw=.8)
    ax.set_xlim(min(-.06, coef["Hệ số"].min() - .04), max(.45, coef["Hệ số"].max() + .08))
    ax.set_xlabel("Hệ số hồi quy chuẩn hóa")
    ax.xaxis.set_major_formatter(lambda x, pos: vi_num(x, 1))
    title(ax, "Thay đổi thiết kế có hệ số chuẩn hóa lớn nhất", f"Mô hình có R² = {vi_num(r2, 3)}; kết quả không được dùng để khẳng định quan hệ nhân quả")
    clean(ax, "x")
    save(fig, "05_thanh_he_so_chuan_hoa.png")

    # 06 — cột ngang theo loại công trình và trạng thái thay đổi
    seg = d.pivot_table(index="Loại công trinh", columns="Thiết kế", values="Số ngày chậm", aggfunc="mean").reindex(columns=["Không thay đổi", "Có thay đổi"])
    seg = seg.sort_values("Có thay đổi")
    fig, ax = plt.subplots(figsize=(11.5, 6.2))
    y = np.arange(len(seg)); h=.34
    b1=ax.barh(y-h/2, seg["Không thay đổi"], height=h, color=GRAY, label="Không thay đổi")
    b2=ax.barh(y+h/2, seg["Có thay đổi"], height=h, color=RED, label="Có thay đổi")
    for bars in [b1,b2]:
        for bar in bars: ax.text(bar.get_width()+.2, bar.get_y()+bar.get_height()/2, vi_num(bar.get_width()), va="center", fontsize=9)
    ax.set_yticks(y, seg.index); ax.axvline(0,color=NAVY,lw=.8); ax.set_xlabel("Số ngày chậm trung bình")
    ax.legend(frameon=False, ncol=2, loc="lower right")
    title(ax, "Nhóm có thay đổi thiết kế trễ hơn ở mọi loại công trình", "So sánh số ngày chậm trung bình trong từng loại công trình")
    clean(ax, "x")
    save(fig, "06_cot_nhom_loai_cong_trinh.png")

    # 07 — year trends by design status
    yr = d.groupby(["Năm bắt đầu", "Thiết kế"]).agg(cham_tb=("Số ngày chậm", "mean")).reset_index()
    fig, ax = plt.subplots(figsize=(10.5, 5.5))
    for label, color, marker in [("Không thay đổi", GRAY, "o"), ("Có thay đổi", RED, "s")]:
        p = yr[yr["Thiết kế"] == label]
        ax.plot(p["Năm bắt đầu"], p.cham_tb, color=color, marker=marker, lw=2.6, markersize=7, label=label)
        for xv, yv in zip(p["Năm bắt đầu"], p.cham_tb):
            ax.text(xv, yv + (.75 if label == "Có thay đổi" else -.75), vi_num(yv), ha="center", va="bottom" if label == "Có thay đổi" else "top", color=color, fontsize=9)
    ax.axhline(0, color=NAVY, lw=.8); ax.set_xticks(sorted(d["Năm bắt đầu"].dropna().unique()))
    ax.set_ylim(-2.8, 10.7)
    ax.set_xlabel("Năm bắt đầu"); ax.set_ylabel("Số ngày chậm trung bình")
    ax.legend(frameon=False, ncol=2, loc="center", bbox_to_anchor=(.5, .47))
    title(ax, "Khoảng cách giữa hai nhóm được duy trì qua các năm", "Các dự án được nhóm theo năm bắt đầu")
    clean(ax, "y")
    save(fig, "07_duong_theo_nam.png")

    # 08 — ranked cause bars; design is largest without erasing other causes
    cause = d[d["Số ngày chậm"] > 0]["Nguyên nhân chậm tiến độ"].fillna("Chưa xác định").value_counts()
    cause = cause[cause > 0].sort_values(ascending=False)
    shares = cause / cause.sum()
    fig, ax = plt.subplots(figsize=(11.5, 6.2))
    plot = shares.sort_values()
    colors = [RED if name == "Thay đổi thiết kế" else "#BCC6CA" for name in plot.index]
    bars = ax.barh(plot.index, plot.values * 100, color=colors, height=.62)
    for bar, name, share in zip(bars, plot.index, plot.values):
        ax.text(
            bar.get_width() + .7, bar.get_y() + bar.get_height()/2,
            f"{vi_num(share * 100)}% · {vi_int(cause[name])} dự án", va="center",
            color=RED if name == "Thay đổi thiết kế" else NAVY,
            fontweight="bold" if name == "Thay đổi thiết kế" else "normal",
        )
    ax.set_xlim(0, max(45, shares.max() * 100 + 7)); ax.set_xlabel("Tỷ trọng trong các dự án bị trễ")
    design_cause_share = shares.get("Thay đổi thiết kế", 0)
    title(
        ax, "Thay đổi thiết kế là nguyên nhân đơn lẻ lớn nhất",
        f"Chiếm {vi_num(design_cause_share * 100)}% số dự án trễ; {vi_num((1-design_cause_share) * 100)}% còn lại đến từ sáu nhóm nguyên nhân khác",
    )
    clean(ax, "x")
    save(fig, "08_thanh_nguyen_nhan.png")

    # 09 — cột tỷ lệ trễ theo mức thay đổi
    fig, ax = plt.subplots(figsize=(11.5, 6))
    order = ["0", "1–2", "3–5", ">5"]
    rate = d.groupby("Nhóm phân phối", observed=False).agg(ty_le=("Số ngày chậm", lambda s:(s>0).mean()*100), n=("STT","count")).reindex(order)
    bars=ax.bar(order, rate.ty_le, color=[GRAY,"#D8AAA5","#CC746B",RED], width=.65)
    for bar, val, n in zip(bars, rate.ty_le, rate.n):
        ax.text(bar.get_x()+bar.get_width()/2, val+2, f"{vi_num(val)}%\nn={vi_int(n)}", ha="center", fontweight="bold")
    ax.set_ylim(0,105); ax.set_xlabel("Số lần thay đổi thiết kế"); ax.set_ylabel("Tỷ lệ dự án bị trễ (%)")
    title(ax, "Tỷ lệ trễ tăng theo số lần thay đổi thiết kế", "Tỷ lệ được tính riêng trong từng nhóm; cỡ mẫu ghi trên mỗi cột")
    clean(ax, "y")
    save(fig, "09_cot_ty_le_tre_theo_muc.png")

    # 10 — cột trước và sau
    positive = d["Số ngày chậm"].clip(lower=0)
    recovered = np.minimum(positive, np.minimum(d["Số lần thay đổi thiết kế"], 1) * max(slope, 0)).sum()
    before = positive.sum(); after = before - recovered
    fig, ax = plt.subplots(figsize=(10.5, 5.7))
    bars=ax.bar([0,1], [before,after], color=[RED,GREEN], width=.58)
    for bar,val in zip(bars,[before,after]): ax.text(bar.get_x()+bar.get_width()/2,val+1800,vi_int(val),ha="center",fontweight="bold",fontsize=13)
    ax.annotate(f"Giảm {vi_int(recovered)} ngày\n({vi_num(recovered/before*100)}%)", xy=(1,after), xytext=(.5,before*1.06), ha="center", color=NAVY, fontweight="bold", arrowprops=dict(arrowstyle="->",color=NAVY,lw=1.5))
    ax.set_xticks([0,1], ["Hiện tại", "Sau khi giảm\nmột lần thay đổi"])
    ax.set_ylabel("Tổng số ngày chậm"); ax.set_ylim(0, before*1.16)
    title(ax, "Giảm một lần thay đổi có thể làm giảm tổng số ngày chậm", "Phân tích độ nhạy, không phải dự báo hoặc cam kết hiệu quả")
    clean(ax, "y")
    save(fig, "10_cot_truoc_sau.png")
    return slope, corr, r2, before, recovered, after


## 3. Kiểm tra dữ liệu đầu vào

In [ ]:
d = load_data()
print(f"Số dòng: {vi_int(len(d))}")
print(f"Tỷ lệ trễ hạn: {vi_num((d['Số ngày chậm'] > 0).mean() * 100)}%")
print(f"Thiếu dữ liệu ở biến trọng tâm: {d['Số lần thay đổi thiết kế'].isna().sum()}")
display(d[['Số lần thay đổi thiết kế', 'Số ngày chậm', 'Số ngày mưa',
           'Số lần thiếu vật tư', 'Giá trị hợp đồng (Tỷ VNĐ)']].describe().round(2))


## 4. Tạo 10 biểu đồ và tóm tắt kết quả

In [ ]:
stats = make_charts(d)
slope, corr, r2, before, recovered, after = stats
print(f"Độ dốc mô tả: {vi_num(slope)} ngày/lần thay đổi")
print(f"Tương quan: {vi_num(corr, 3)}; R² mô tả: {vi_num(r2, 3)}")
print(f"Phân tích độ nhạy: số ngày chậm có thể giảm {vi_int(recovered)} ngày ({vi_num(recovered/before*100)}%)")
print(f"Đã lưu {len(list(CHARTS.glob('*.png')))} hình vào: {CHARTS}")


## 5. Giới hạn diễn giải

- Hồi quy, tương quan và phân tích độ nhạy là các công cụ mô tả, không tự động chứng minh quan hệ nhân quả.
- Hiệu quả của mốc chốt thiết kế cần được đánh giá bằng chương trình thí điểm, sử dụng số liệu trước triển khai và nhóm so sánh phù hợp.
